# Januar batch: Smart_download + Script - azimuth/range uden vending

In [4]:
import os
import re
import colorsys
from pathlib import Path
from datetime import datetime
from PIL import ImageFont

import requests
import rasterio
import pandas as pd
from PIL import Image, ImageDraw
import xml.etree.ElementTree as ET
import numpy as np
from scipy.spatial import cKDTree
from IPython.display import clear_output
import geopandas as gpd
from shapely import wkt
import trino
from trino import exceptions as trino_exceptions
from trino.auth import OAuth2Authentication
# AENDRING AZIMUTH/RANGE: GCPTransformer bruges kun til lon/lat -> original SAR row/col.
from rasterio.transform import GCPTransformer


In [ ]:
# ----------
# USER INPUT
# ----------

# Enter your OpenSky Trino username, the code will later open the webbrowser to access trino, where you can log in.
OPENSKY_USER = ""

# Copernicus Data Space Ecosystem login
# Create an account at: https://dataspace.copernicus.eu/
USERNAME = r"" # insert username here
PASSWORD = r"" # insert password here

# Modify the date to the range of SAR images you want to download. THe default is set to download one image on 20/09/2025 for BAW14K & DAL 176
DATE_FROM = "2025-09-20T00:00:00.000Z"
DATE_TO   = "2025-09-21T20:00:00.000Z"

# Define desired orbit direction, we only used Descending in our dataset
ORBIT_DIRECTION = "DESCENDING"  # "ASCENDING" or "DESCENDING"

# Defined search area bounds on the north-west coast of Ireland
BBOX = (-10.310, 54.169, -10.002, 54.349)

# Making a folder called 2025, for all the images in 2025
BASE_DIR = Path.cwd() / "2025"
print(BASE_DIR)

DOWNLOAD_DIR = BASE_DIR / "september_downloads"  # Folder for the raw SAR images, change the name so it fits the date specified above, e.g. "september_downloads".

# AZIMUTH/RANGE: Folder where te images will be stored.
IMAGE_DIR = BASE_DIR / "september_images_azimuth_range"

DOWNLOAD_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

# Values of inclination depending on orbit
inclination_ascending = 98.18
inclination_descending = 81.82

# Inclination is selected automatically based on ORBIT_DIRECTION
inclination = inclination_ascending if ORBIT_DIRECTION == "ASCENDING" else inclination_descending

# Defined search area bounds on the north-west coast of Ireland
polygon_wkt = """POLYGON((
-7.590541 54.824537,
-11.678641 55.245177,
-12.069724 53.751265,
-8.125319 53.334224,
-7.590541 54.824537
))"""

buffer = 0.1

# These can be disabled if downloads already exist or images have already been generated.
RUN_DOWNLOAD = True
RUN_IMAGE_GENERATION = True
SKIP_EXISTING_DOWNLOADS = True
SKIP_EXISTING_IMAGES = True

c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\SAR_image_BAW14K\2025


In [ ]:
# ---------------------------
# SMART_DOWNLOAD - This code downloads the Sentinel-1 SAR images from copernicus browser (https://dataspace.copernicus.eu/). 
# The 16-bit images are quite big, so it takes a few minutes per image. 
# If you have already downloaded the image for a certain date previously and it is located in the folder, the code will detect this and let you know.
# ---------------------------

def get_token(username, password):
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password"
    }
    response = requests.post(url, data=data)
    response.raise_for_status()
    return response.json()["access_token"]


def search_sentinel1_products(headers=None):
    print("Searching Sentinel-1 products...")

    minx, miny, maxx, maxy = BBOX
    wkt_bbox = f"POLYGON(({minx} {miny},{maxx} {miny},{maxx} {maxy},{minx} {maxy},{minx} {miny}))"

    filter_query = (
        f"Collection/Name eq 'SENTINEL-1' and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt_bbox}') and "
        f"ContentDate/Start gt {DATE_FROM} and "
        f"ContentDate/Start lt {DATE_TO} and "
        "Attributes/OData.CSC.StringAttribute/any(att: "
        f"att/Name eq 'orbitDirection' and att/Value eq '{ORBIT_DIRECTION}') and "
        "Attributes/OData.CSC.StringAttribute/any(att: "
        "att/Name eq 'productType' and att/Value eq 'IW_GRDH_1S')"
    )

    search_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    params = {
        "$filter": filter_query,
        "$orderby": "ContentDate/Start asc",
        "$top": 1000
    }

    resp = requests.get(search_url, params=params)
    resp.raise_for_status()
    data = resp.json()["value"]

    if len(data) == 0:
        print("No files found.")
    else:
        print(f"{len(data)} files found:")
    for product in data:
        print(product["Name"])

    return data


def download_file(url, path, headers):
    if SKIP_EXISTING_DOWNLOADS and path.exists() and path.stat().st_size > 0:
        print("Skipping existing:", path.name)
        return

    print(f"Downloading {path.name}...")
    r = requests.get(url, headers=headers, stream=True)
    r.raise_for_status()

    tmp_path = path.with_suffix(path.suffix + ".part")
    with open(tmp_path, "wb") as f:
        for chunk in r.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)

    tmp_path.replace(path)
    print("Saved to:", path)


def product_dir_has_existing_scene_files(product_dir):
    if not product_dir.exists() or not product_dir.is_dir():
        return False

    files = [p for p in product_dir.iterdir() if p.is_file() and p.stat().st_size > 0]
    if len(files) < 2:
        return False

    has_vv_xml = any("vv" in p.name.lower() and p.suffix.lower() == ".xml" for p in files)
    has_vv_tiff = any("vv" in p.name.lower() and p.suffix.lower() in (".tif", ".tiff") for p in files)
    return has_vv_xml and has_vv_tiff


def existing_scene_dirs_in_download_dir(download_dir):
    if not download_dir.exists() or not download_dir.is_dir():
        return []

    scene_dirs = []
    if product_dir_has_existing_scene_files(download_dir):
        scene_dirs.append(download_dir)

    for product_dir in sorted(download_dir.glob("*.SAFE")):
        if product_dir_has_existing_scene_files(product_dir):
            scene_dirs.append(product_dir)

    return scene_dirs
def download_vv_files_for_products(data, headers):
    DOWNLOAD_BASE_URL = "https://download.dataspace.copernicus.eu/odata/v1"

    for product in data:
        safe_name = product["Name"]
        product_id = product["Id"]

        product_dir = DOWNLOAD_DIR / safe_name

        print(f"\n--- Processing: {safe_name} ---")

        if SKIP_EXISTING_DOWNLOADS and product_dir_has_existing_scene_files(product_dir):
            print("Using existing SAFE folder; no download needed:", product_dir)
            continue

        product_dir.mkdir(exist_ok=True)

        annotation_url = (
            f"{DOWNLOAD_BASE_URL}/Products({product_id})/"
            f"Nodes({safe_name})/"
            f"Nodes(annotation)/Nodes"
        )

        resp_annot = requests.get(annotation_url, headers=headers)
        resp_annot.raise_for_status()
        annotation_json = resp_annot.json()
        annotation_files = annotation_json.get("value", annotation_json.get("result", []))

        xml_file = None
        for f in annotation_files:
            name = f["Name"]
            if "vv" in name.lower() and name.endswith(".xml"):
                xml_file = name
                break

        measurement_url = (
            f"{DOWNLOAD_BASE_URL}/Products({product_id})/"
            f"Nodes({safe_name})/"
            f"Nodes(measurement)/Nodes"
        )

        resp_meas = requests.get(measurement_url, headers=headers)
        resp_meas.raise_for_status()
        measurement_json = resp_meas.json()
        measurement_files = measurement_json.get("value", measurement_json.get("result", []))

        tiff_file = None
        for f in measurement_files:
            name = f["Name"]
            if "vv" in name.lower() and name.endswith(".tiff"):
                tiff_file = name
                break

        print("Selected VV files:")
        print("XML :", xml_file)
        print("TIFF:", tiff_file)

        if xml_file:
            xml_url = (
                f"{DOWNLOAD_BASE_URL}/Products({product_id})/"
                f"Nodes({safe_name})/"
                f"Nodes(annotation)/"
                f"Nodes({xml_file})/$value"
            )
            download_file(xml_url, product_dir / xml_file, headers)
        else:
            print("No matching VV XML file found for this product.")

        if tiff_file:
            tiff_url = (
                f"{DOWNLOAD_BASE_URL}/Products({product_id})/"
                f"Nodes({safe_name})/"
                f"Nodes(measurement)/"
                f"Nodes({tiff_file})/$value"
            )
            download_file(tiff_url, product_dir / tiff_file, headers)
        else:
            print("No matching VV TIFF file found for this product.")

        print(f"Finished processing product: {product_id}\n")


if RUN_DOWNLOAD:
    existing_scene_dirs = existing_scene_dirs_in_download_dir(DOWNLOAD_DIR)
    if existing_scene_dirs:
        print(f"Using {len(existing_scene_dirs)} existing SAFE folder(s) in {DOWNLOAD_DIR}")
        print("No Copernicus login/search/download needed; ADS-B will still be fetched below.")
    else:
        print("No existing VV XML/TIFF scene files found in", DOWNLOAD_DIR)
        print("Authenticating...")
        token = get_token(USERNAME, PASSWORD)
        headers = {"Authorization": f"Bearer {token}"}
        products = search_sentinel1_products(headers)
        download_vv_files_for_products(products, headers)
else:
    print("RUN_DOWNLOAD is False - using files already in", DOWNLOAD_DIR)


Using 1 existing SAFE folder(s) in c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\SAR_image_BAW14K\2025\september_downloads
No Copernicus login/search/download needed; ADS-B will still be fetched below.


In [ ]:
# ---------------------------
# Helper functions
#
# This section contains all functions required for the processing chain:
# 1. Query aircraft tracks from OpenSky within a given time window and area.
# 2. Find Sentinel-1 scenes and extract acquisition times from filenames.
# 3. Build an aircraft dataframe for each SAR scene by:
#    - retrieving OpenSky aircraft positions,
#    - interpolating aircraft trajectories over the SAR acquisition period,
#    - filtering aircraft inside the area of interest,
#    - matching aircraft positions to the Sentinel-1 geolocation grid,
#    - computing slant range and incidence angle,
#    - calculating the expected Doppler-induced azimuth displacement.
#
# The resulting dataframe is later used to compare aircraft positions
# with detections in Sentinel-1 SAR imagery.
# ---------------------------

def query_opensky(cur, t_begin_str, t_end_str, polygon_wkt, buffer_deg=0.1, limit=10000):
    """
    Same query function as used in Script.ipynb.
    """
    t_begin = datetime.fromisoformat(t_begin_str.replace("Z", "+00:00"))
    t_end   = datetime.fromisoformat(t_end_str.replace("Z", "+00:00"))

    t_start_epoch = int(t_begin.timestamp()) - 300
    t_stop_epoch  = int(t_end.timestamp()) + 300
    hour_partition = t_start_epoch - (t_start_epoch % 3600)

    query = f"""
    SELECT *
    FROM state_vectors_data4
    WHERE hour = {hour_partition}
      AND time BETWEEN {t_start_epoch} AND {t_stop_epoch}
      AND ST_Contains(
            ST_Buffer(
                ST_GeometryFromText('{polygon_wkt}'),
                {buffer_deg}
            ),
            ST_Point(lon, lat)
          )
    LIMIT {limit}
    """

    cur.execute(query)
    rows = cur.fetchall()
    return rows


def scene_times_from_tiff_name(tiff_path):
    match = re.search(r"-(\d{8}t\d{6})-(\d{8}t\d{6})-", tiff_path.name.lower())
    if not match:
        raise ValueError(f"Could not parse scene time from {tiff_path.name}")
    start = datetime.strptime(match.group(1), "%Y%m%dt%H%M%S")
    stop = datetime.strptime(match.group(2), "%Y%m%dt%H%M%S")
    return start.strftime("%Y-%m-%dT%H:%M:%SZ"), stop.strftime("%Y-%m-%dT%H:%M:%SZ"), start.strftime("%Y-%m-%d")


def find_scene_pairs(download_dir):
    # Search for XML/TIFF pairs in the download directory
    candidate_dirs = [download_dir] if download_dir.name.endswith(".SAFE") else sorted(download_dir.glob("*.SAFE"))
    for product_dir in candidate_dirs:
        xml_files = list(product_dir.glob("*vv*.xml"))
        tiff_files = list(product_dir.glob("*vv*.tiff")) + list(product_dir.glob("*vv*.tif"))
        if xml_files and tiff_files:
            yield product_dir, xml_files[0], tiff_files[0]
        else:
            print("Skipping incomplete scene:", product_dir.name)


def prepare_aircraft_dataframe(cur, t_start, t_slut, xml_path):
    # This function combines the aircraft-processing cells so they can be reused automatically for each SAR scene.

    rows = query_opensky(
        cur,
        t_begin_str=t_start,
        t_end_str=t_slut,
        polygon_wkt=polygon_wkt,
        buffer_deg=buffer,
        limit=10000
    )

    if len(rows) == 0:
        print("No OpenSky rows found in time/polygon window.")
        return pd.DataFrame()

    colnames = [desc[0] for desc in cur.description]
    df = pd.DataFrame(rows, columns=colnames)
    if df.empty:
        return df

    c = 299792458
    v_sat = 7400
    azimuth_pixel_spacing = 10

    lat_list = []
    lon_list = []
    slant_list = []
    inc_list = []

    df_place_look = df[df["onground"] == False].copy()
    if df_place_look.empty:
        return df_place_look

    t_sar_start = pd.to_datetime(t_start, utc=True)
    t_sar_end   = pd.to_datetime(t_slut, utc=True)

    df_place_look["time"] = pd.to_datetime(df_place_look["time"], unit="s", utc=True)
    df_place_look = df_place_look.drop_duplicates(subset=["icao24", "lon", "lat"], keep="first")
    df_place_look = df_place_look.sort_values(["icao24", "time"])

    all_planes = []
    for icao24, grp in df_place_look.groupby("icao24"):
        grp = grp.copy().sort_values("time")
        grp = grp.set_index("time")

        for t in [t_sar_start, t_sar_end]:
            if t not in grp.index:
                grp.loc[t] = np.nan

        grp = grp.sort_index()
        grp["callsign"] = grp["callsign"].ffill().bfill()
        grp["icao24"] = icao24

        grp["heading_rad"] = np.deg2rad(grp["heading"])
        grp["heading_sin"] = np.sin(grp["heading_rad"])
        grp["heading_cos"] = np.cos(grp["heading_rad"])

        cols = ["lat", "lon", "velocity", "heading_sin", "heading_cos"]
        grp[cols] = grp[cols].interpolate(method="time", limit_area="inside")

        grp["heading"] = np.rad2deg(np.arctan2(grp["heading_sin"], grp["heading_cos"]))
        grp["heading"] = (grp["heading"] + 360) % 360

        grp = grp.reset_index()
        all_planes.append(grp)

    if len(all_planes) == 0:
        return pd.DataFrame()

    df_place_look_interp = pd.concat(all_planes, ignore_index=True)
    df_place_look_interp = df_place_look_interp.copy()

    gdf = gpd.GeoDataFrame(
        df_place_look_interp,
        geometry=gpd.points_from_xy(df_place_look_interp["lon"], df_place_look_interp["lat"]),
        crs="EPSG:4326"
    )

    polygon = wkt.loads(polygon_wkt)

    df_filtered = gdf[
        (gdf["time"] >= t_sar_start) &
        (gdf["time"] <= t_sar_end) &
        (gdf.within(polygon))
    ].copy()

    if df_filtered.empty:
        return df_filtered

    # TRACKBOX:
    # Use only the interpolated start and end positions of each aircraft.
    # This reproduces the January_batch logic where the track box
    # is defined using a line between p1 and p2.
    df_filtered = df_filtered.sort_values(["icao24", "time"]).copy()
    track_rows = []
    for _, grp in df_filtered.groupby("icao24", sort=False):
        grp = grp.sort_values("time")
        if len(grp) >= 2:
            track_rows.append(grp.iloc[[0, -1]])
    if not track_rows:
        return pd.DataFrame()
    df_filtered = pd.concat(track_rows, ignore_index=True)

    tree_xml = ET.parse(xml_path)
    root = tree_xml.getroot()
    points = root.findall(".//geolocationGridPoint")

    for p in points:
        lat_list.append(float(p.find("latitude").text))
        lon_list.append(float(p.find("longitude").text))
        slant_list.append(float(p.find("slantRangeTime").text))
        inc_list.append(float(p.find("incidenceAngle").text))

    lat_arr = np.array(lat_list)
    lon_arr = np.array(lon_list)
    slant_arr = np.array(slant_list)
    inc_arr = np.array(inc_list)
    coords = np.column_stack((lat_arr, lon_arr))
    tree_geo = cKDTree(coords)

    aircraft_coords = np.column_stack((df_filtered["lat"], df_filtered["lon"]))
    distance, idx = tree_geo.query(aircraft_coords)
    t_slant = slant_arr[idx]
    incidence = inc_arr[idx]

    R_slant = c * t_slant / 2

    df_filtered["slant_range_time"] = t_slant
    df_filtered["R_slant_m"] = R_slant
    df_filtered["incidence_angle"] = incidence

    def offset_doppler1(sog, cog, R_slant, lat_aircraft, incidence_angle, v_sat, inclination_angle):
        lat_aircraft = np.deg2rad(lat_aircraft)
        cog = np.deg2rad(cog)
        incidence_angle = np.deg2rad(incidence_angle)
        inclination_angle = np.deg2rad(inclination_angle)

        cog_AIS_proj = cog - np.arcsin(
            np.cos(inclination_angle) / np.cos(lat_aircraft)
        )

        return sog * np.sin(cog_AIS_proj) * R_slant * np.sin(incidence_angle) / v_sat

    doppler_offset = offset_doppler1(
        sog=df_filtered["velocity"].values,
        cog=df_filtered["heading"].values,
        R_slant=df_filtered["R_slant_m"].values,
        lat_aircraft=df_filtered["lat"].values,
        incidence_angle=df_filtered["incidence_angle"].values,
        v_sat=v_sat,
        inclination_angle=inclination
    )

    df_filtered["doppler_offset_m"] = doppler_offset
    df_filtered["doppler_offset_px"] = df_filtered["doppler_offset_m"] / azimuth_pixel_spacing

    return df_filtered

In [22]:
# ---------------------------
# IMAGE GENERATION AND VISUALISATION IN ORIGINAL SAR AZIMUTH/RANGE GEOMETRY
#
# This function loads a Sentinel-1 SAR image and overlays aircraft tracks
# obtained from OpenSky. Aircraft positions are projected directly into the
# original SAR image geometry using the Sentinel-1 geolocation grid.
#
# For each aircraft:
# - The GPS position is converted to SAR pixel coordinates.
# - The Doppler-induced azimuth displacement is applied.
# - The flight heading is projected into SAR image coordinates.
# - A Doppler box, heading arrow and aircraft label are drawn.
#
# The SAR image is displayed in its original azimuth/range geometry:
# - No reprojection
# - No rotation
# - No flipping
# - No map projection
#
# The final annotated image is saved as a JPEG file.
# ---------------------------

def draw_sar_image(tiff_path, df_filtered, output_jpg):
    POINT_SIZE = 30
    DOPPLER_WIDTH = 500
    OUTLINE_WIDTH = 10
    HEADING_LEN = 200
    TEXT_DIS = 80
    FONT_SIZE = 80

    font = ImageFont.truetype("arial.ttf", FONT_SIZE)

    def lonlat_to_sar_pixel(transformer, lon, lat):
        # Convert geographic coordinates (longitude/latitude)
        # directly to original SAR image line/pixel coordinates.
        row, col = transformer.rowcol(lon, lat, op=lambda value: value)
        return float(col), float(row)

    def heading_to_sar_unit(transformer, lon, lat, heading, x0, y0, step_m=1000.0):
        # Convert the true geographic heading into a local SAR pixel vector.
        heading_rad = np.deg2rad(heading)
        lat_rad = np.deg2rad(lat)
        d_lat = (step_m * np.cos(heading_rad)) / 111320.0
        d_lon = (step_m * np.sin(heading_rad)) / (111320.0 * np.cos(lat_rad))
        x1, y1 = lonlat_to_sar_pixel(transformer, lon + d_lon, lat + d_lat)
        dx, dy = x1 - x0, y1 - y0
        length = float(np.hypot(dx, dy))
        if not np.isfinite(length) or length == 0:
            return 0.0, -1.0
        return dx / length, dy / length

    with rasterio.open(tiff_path) as src:
        # Read the raw SAR TIFF directly.
        sar_data = src.read(1)
        gcps, _ = src.gcps
        transformer = GCPTransformer(gcps)

    # Applying linear percentile scaling directly to the raw SAR image.
    data = sar_data
    vmin, vmax = np.percentile(data, [2, 98])
    if vmax <= vmin:
        data_8bit = np.zeros_like(data, dtype=np.uint8)
    else:
        data_8bit = np.clip((data - vmin) / (vmax - vmin) * 255, 0, 255).astype(np.uint8)

    final_img = Image.fromarray(np.stack([data_8bit] * 3, axis=-1))
    draw = ImageDraw.Draw(final_img)

    try:
        plane_data = {}

        for row in df_filtered.itertuples():
            px, py = lonlat_to_sar_pixel(transformer, row.lon, row.lat)

            # Applying the Doppler displacement directly in the SAR azimuth direction.
            # The range coordinate is preserved.
            # Positive Doppler offsets move downward in the image.
            pgx = px
            pgy = py + row.doppler_offset_px

            heading_vector = heading_to_sar_unit(
                transformer,
                row.lon,
                row.lat,
                row.heading,
                px,
                py,
            )

            p_id = str(getattr(row, 'callsign', getattr(row, 'icao24', 'Plane'))).strip()

            if p_id == "" or p_id.lower() == "nan":
                p_id = str(getattr(row, 'icao24', 'Plane'))

            if p_id not in plane_data:
                plane_data[p_id] = {
                    'gps': [],
                    'doppler': [],
                    'heading_vector': heading_vector
                }

            plane_data[p_id]['gps'].append((int(px), int(py)))
            plane_data[p_id]['doppler'].append((int(pgx), int(pgy)))

    finally:
        transformer.close()

    d_ellipse = draw.ellipse
    d_line = draw.line
    d_polygon = draw.polygon
    d_text = draw.text

    for i, (p_id, data) in enumerate(plane_data.items()):
        hue = (i * 0.137) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 1.0, 1.0)
        colour = (int(r * 255), int(g * 255), int(b * 255))

        gps, dopp = data['gps'], data['doppler']
        p1_g, p2_g = gps[0], gps[-1]
        p1_d, p2_d = dopp[0], dopp[-1]

        for p in (p1_g, p2_g):
            d_ellipse(
                [p[0] - POINT_SIZE, p[1] - POINT_SIZE,
                 p[0] + POINT_SIZE, p[1] + POINT_SIZE],
                fill=colour
            )

        # Use the precomputed SAR heading vector
        ux, uy = data['heading_vector']

        # Compute the heading arrow end point
        hx = p1_g[0] + (HEADING_LEN * ux)
        hy = p1_g[1] + (HEADING_LEN * uy)

        # Direction vector
        sub_x = ux
        sub_y = uy

        # Perpendicular vector
        per_x = -sub_y
        per_y = sub_x

        ex, ey = DOPPLER_WIDTH * sub_x, DOPPLER_WIDTH * sub_y
        ox, oy = DOPPLER_WIDTH * per_x, DOPPLER_WIDTH * per_y

        # Explicit conversion to standard Python integers
        corners = [
            (int(p1_d[0] - ex + ox), int(p1_d[1] - ey + oy)),
            (int(p2_d[0] + ex + ox), int(p2_d[1] + ey + oy)),
            (int(p2_d[0] + ex - ox), int(p2_d[1] + ey - oy)),
            (int(p1_d[0] - ex - ox), int(p1_d[1] - ey - oy))
        ]

        d_polygon(corners, outline=colour, width=OUTLINE_WIDTH)
        d_line([p1_g, (int(hx), int(hy))], fill=colour, width=15)
        d_text(
            (p1_g[0] + 80, p1_g[1] - TEXT_DIS),
            p_id,
            fill=(255, 255, 255),
            font=font
        )

    clear_output(wait=True)
    print("Percentage done: {:.2f}%".format((i + 1) / len(plane_data) * 100))

    final_img.save(output_jpg, "JPEG", quality=95)

    clear_output(wait=True)
    print(f"Successfully saved: {output_jpg}")

In [24]:
# ---------------------------
# PROCESS ALL DOWNLOADED SCENES AND GENERATE SAR OVERLAY IMAGES
#
# This section loops through all downloaded Sentinel-1 scenes and:
# 1. Extracts acquisition times from the scene filename.
# 2. Queries OpenSky aircraft data for the SAR acquisition period.
# 3. Interpolates aircraft tracks and computes Doppler offsets.
# 4. Projects aircraft positions into the SAR image geometry.
# 5. Draws aircraft tracks, Doppler boxes and heading vectors.
# 6. Saves the final annotated image as a JPEG.
#
# Scenes without relevant aircraft are skipped.
# Existing images can optionally be reused without regeneration.
# ---------------------------

if RUN_IMAGE_GENERATION:
    conn = trino.dbapi.connect(
        host="trino.opensky-network.org",
        port=443,
        user=OPENSKY_USER,
        http_scheme="https",
        auth=OAuth2Authentication(),
        catalog="minio",
        schema="osky",
    )

    cur = conn.cursor()
    made_images = []
    skipped_scenes = []

    for product_dir, xml_path, tiff_path in find_scene_pairs(DOWNLOAD_DIR):

        t_start, t_slut, scene_date = scene_times_from_tiff_name(tiff_path)
        output_jpg = IMAGE_DIR / f"{scene_date}_trackbox_rawdoppler_rect.jpg"

        print(f"\n--- Building image for {scene_date}: {product_dir.name} ---")
        print("XML :", xml_path)
        print("TIFF:", tiff_path)
        print("t_start:", t_start)
        print("t_slut :", t_slut)

        if SKIP_EXISTING_IMAGES and output_jpg.exists() and output_jpg.stat().st_size > 0:
            print("Skipping existing image:", output_jpg)
            made_images.append(output_jpg)
            continue

        try:
            df_filtered = prepare_aircraft_dataframe(cur, t_start, t_slut, xml_path)

        except trino_exceptions.TrinoAuthError as err:
            print("OpenSky/Trino login failed before image generation.")
            print("If a browser login URL was shown above, open it and approve the login, then run this cell again.")
            print("Original error:", err)
            break

        if df_filtered.empty:

            print("No aircraft within SAR polygon/time; no image saved for", scene_date)
            skipped_scenes.append((scene_date, product_dir.name))
            continue

        print(df_filtered)

        draw_sar_image(tiff_path, df_filtered, output_jpg)

        made_images.append(output_jpg)

    cur.close()
    conn.close()

    print("\nDone.")

    print("Images made:")
    for path in made_images:
        print(path)

    print("\nScenes without image:")
    for scene_date, name in skipped_scenes:
        print(scene_date, name)

else:
    print("RUN_IMAGE_GENERATION is False - image generation skipped.")

Successfully saved: c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\SAR_image_BAW14K\2025\september_images_azimuth_range\2025-09-20_trackbox_rawdoppler_rect.jpg

Done.
Images made:
c:\Users\Johannes Pielmeier\OneDrive\Education\DTU 2024 - 2029\Fagprojekt\SAR_image_BAW14K\2025\september_images_azimuth_range\2025-09-20_trackbox_rawdoppler_rect.jpg

Scenes without image:
